<a href="https://colab.research.google.com/github/viettungvuong/HR-Information/blob/main/HR_Information.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install datasets


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from datasets import load_dataset

dataset_name = "tinixai/vietnamese-job-descriptions"
dataset = load_dataset(dataset_name)

df = dataset['train'].to_pandas()
print(df.head())

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


   id                                          job_title  \
0   1                Java Software Engineer - MSCV 233-2   
1   2  Software Engineer (Backend/Frontend) - Tiếng T...   
2   3  Lead Software Engineer – Desktop (Remote, Engl...   
3   4              Lâp Trình Viên/ Software Engineer/ IT   
4   5                        Software Engineer (Fintech)   

                                        company_name         salary  \
0                       CÔNG TY TNHH QUICK VIỆT NAM  26 - 36 triệu   
1                          CÔNG TY TNHH SONIC FUSION  15 - 35 triệu   
2  CÔNG TY TNHH PHẦN MỀM GIÁ TRỊ TÍNH TOÁN VÀ ỨNG...       4000 usd   
3  CÔNG TY TNHH ĐẦU TƯ THƯƠNG MẠI VÀ DỊCH VỤ PHÁT...  10 - 15 triệu   
4                                 CÔNG TY TNHH CASSO  14 - 20 triệu   

      location        job_type job_industry experience_level education_level  \
0  Hồ Chí Minh  Toàn thời gian  IT phần mềm            3 năm        Cao đẳng   
1  Hồ Chí Minh  Toàn thời gian  IT phần mềm       Dư

### Step 1: Entity Extraction Strategy
To build the Knowledge Graph, you'll iterate through your dataset and pass the unstructured text to an LLM to extract structured entities and relationships. Here is an example of how you would format the prompt for a single job row.

In [3]:
# Let's take the requirements from the second job in our dataset as an example
sample_job_id = df.iloc[1]['id']
sample_requirement = df.iloc[1]['requirements']

print(f"--- Original Requirement Text (Job ID: {sample_job_id}) ---")
print(sample_requirement)

# This is the prompt template you would send to an LLM (like Gemini or OpenAI)
prompt_template = f"""
You are an expert Knowledge Graph extractor specializing in HR and recruitment.
Analyze the following job requirement text and extract the required skills, programming languages, and tools.

Return the output strictly as a JSON list of relationships, like this:
[{{"source": "Job_{sample_job_id}", "relationship": "REQUIRES_SKILL", "target": "Python"}}]

Text:
{sample_requirement}
"""

print("\n--- Prompt to send to LLM ---")
print(prompt_template)

--- Original Requirement Text (Job ID: 2) ---
Thành thạo ít nhất một trong các ngôn ngữ: Python / Java / NodeJS / PHP Ưu điểm: có kinh nghiệm về AI, Machine Learning hoặc AdTech, đã làm việc với hệ thống data lớn hoặc automation Có khả năng đọc hiểu tài liệu tiếng Trung (HSK4+ là lợi thế) Có kinh nghiệm lập trình từ 1 năm trở lên Có hiểu biết về API, database, hệ thống backend

--- Prompt to send to LLM ---

You are an expert Knowledge Graph extractor specializing in HR and recruitment.
Analyze the following job requirement text and extract the required skills, programming languages, and tools.

Return the output strictly as a JSON list of relationships, like this:
[{"source": "Job_2", "relationship": "REQUIRES_SKILL", "target": "Python"}]

Text:
Thành thạo ít nhất một trong các ngôn ngữ: Python / Java / NodeJS / PHP Ưu điểm: có kinh nghiệm về AI, Machine Learning hoặc AdTech, đã làm việc với hệ thống data lớn hoặc automation Có khả năng đọc hiểu tài liệu tiếng Trung (HSK4+ là lợi thế)

### Step 2: Data Cleaning
Clean data to extract more information

In [4]:
pip install python-dotenv requests


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import json
import requests
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

def extract_structured_data(benefits_text, requirements_text):
    prompt = f"""
    You are an expert HR data structurer. Extract the benefits and requirements from the provided text into clear bullet points.

    CRITICAL RULES:
    1. Return a JSON object with exactly two keys: \"benefits\" (a list of strings) and \"requirements\" (a list of strings).
    2. For \"benefits\", if salary or compensation information is mentioned, it MUST be the very first item in the \"benefits\" list. Anything related to SHUI (Bảo hiểm xã hội) must be second. All of those must be noted with (Salary) and (Social Insurance) tags at the end of the bullet point.
    3. Keep the extracted bullet points concise and in Vietnamese.

    Benefits Text:
    {benefits_text}

    Requirements Text:
    {requirements_text}
    """

    api_key = os.getenv("OPENROUTER_API_KEY")
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }

    try:
        response = requests.post('https://openrouter.ai/api/v1/chat/completions', headers=headers, data=json.dumps({
            "model": "poolside/laguna-m.1:free",
            "messages": [
                {
                    "role": "system",
                    "content": "You are a helpful data extraction assistant that outputs strictly valid JSON."
                },
                {
                    "role": "user",
                    "content": prompt,
                }
            ]
        }))

        response.raise_for_status()
        
        raw_result = response.json()
        content = raw_result.get('choices', [{}])[0].get('message', {}).get('content', '')
        print(f"Extracted successfully with {content}")
        return json.loads(content)
        
    except Exception as e:
        print(f"Unexpected Error during OpenRouter API call: {e}")
        return {"benefits": [], "requirements": []}

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

MAX_WORKERS = 5 

extracted_benefits = [None] * len(df)
extracted_requirements = [None] * len(df)

print(f"Starting parallel extraction with {MAX_WORKERS} workers...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # Submit all tasks
    future_to_index = {
        executor.submit(extract_structured_data, row['benefits'], row['requirements']): i 
        for i, row in df.iterrows()
    }
    
    # Process as they finish
    for future in tqdm(as_completed(future_to_index), total=len(df)):
        index = future_to_index[future]
        try:
            result = future.result()
            extracted_benefits[index] = result.get('benefits', [])
            extracted_requirements[index] = result.get('requirements', [])
        except Exception as e:
            print(f"Row {index} generated an unhandled exception: {e}")
            extracted_benefits[index] = []
            extracted_requirements[index] = []

df['extracted_benefits'] = extracted_benefits
df['extracted_requirements'] = extracted_requirements

print("All jobs completed.")

Starting parallel extraction with 5 workers...


  0%|          | 1/606878 [00:02<463:06:11,  2.75s/it]

{'id': 'gen-1778000271-RVI6fSeXCOQRSHas5jdL', 'object': 'chat.completion', 'created': 1778000271, 'model': 'poolside/laguna-m.1-20260312:free', 'provider': 'Poolside', 'system_fingerprint': None, 'choices': [{'index': 0, 'logprobs': None, 'finish_reason': 'stop', 'native_finish_reason': 'stop', 'message': {'role': 'assistant', 'content': '\n{\n  "benefits": [\n    "Thu nhập hấp dẫn: 15 triệu - 35 triệu (Tuỳ theo năng lực) (Salary)",\n    "Chế độ bảo hiểm và nghỉ phép đầy đủ theo quy định pháp luật (Social Insurance)",\n    "Cơ hội làm việc trong môi trường trẻ trung, sáng tạo và công nghệ mới nhất"\n  ],\n  "requirements": [\n    "Thành thạo ít nhất một trong các ngôn ngữ: Python / Java / NodeJS / PHP",\n    "Ưu điểm: có kinh nghiệm về AI, Machine Learning hoặc AdTech, đã làm việc với hệ thống data lớn hoặc automation",\n    "Có khả năng đọc hiểu tài liệu tiếng Trung (HSK4+ là lợi thế)",\n    "Có kinh nghiệm lập trình từ 1 năm trở lên",\n    "Có hiểu biết về API, database, hệ thống bac

  0%|          | 2/606878 [00:04<364:43:32,  2.16s/it]

{'id': 'gen-1778000271-nJmStnXQO3QM7fneMI58', 'object': 'chat.completion', 'created': 1778000271, 'model': 'poolside/laguna-m.1-20260312:free', 'provider': 'Poolside', 'system_fingerprint': None, 'choices': [{'index': 0, 'logprobs': None, 'finish_reason': 'stop', 'native_finish_reason': 'stop', 'message': {'role': 'assistant', 'content': '\n```json\n{\n  "benefits": [\n    "Mức lương cạnh tranh từ 14.000.000 - 20.000.000 VNĐ + thưởng tết, quà tết, thưởng hiệu suất (Salary)",\n    "BHXH, BHYT theo quy định Nhà nước (Social Insurance)",\n    "Gói khám sức khoẻ toàn diện vào cuối năm",\n    "Du lịch trong và ngoài nước cùng công ty",\n    "Review lương 6 tháng/lần",\n    "Cơ hội lên lead nhanh nếu chứng minh được năng lực",\n    "Hỗ trợ công cụ, thiết bị, data phục vụ nghiên cứu",\n    "Thời gian làm việc: 8h-17h30 Thứ Hai – Thứ Sáu",\n    "Môi trường làm việc startup fintech năng động, sáng tạo"\n  ],\n  "requirements": [\n    "Thành thạo ít nhất một ngôn ngữ Backend: NodeJS / Python / J

...

{'id': 'gen-1778000296-YOG9GdVtFWPAjnfdxmrk', 'object': 'chat.completion', 'created': 1778000296, 'model': 'poolside/laguna-m.1-20260312:free', 'provider': 'Poolside', 'system_fingerprint': None, 'choices': [{'index': 0, 'logprobs': None, 'finish_reason': 'stop', 'native_finish_reason': 'stop', 'message': {'role': 'assistant', 'content': '\n{\n  "benefits": [\n    "Thưởng hàng năm (Salary)",\n    "Môi trường làm việc và cơ sở vật chất khu bếp",\n    "Đào tạo chéo, phát triển sự nghiệp và hỗ trợ nơi làm việc",\n    "Nghỉ phép và kỳ nghỉ",\n    "Phát triển sự nghiệp"\n  ],\n  "requirements": []\n}\n', 'refusal': None, 'reasoning': '\nOkay, let\'s see. The user wants me to extract benefits and requirements from the given text into bullet points in Vietnamese. The JSON should have two keys: "benefits" and "requirements". \n\nFirst, for the benefits. The input mentions "pantry area work environment and facilities, yearly bonus financial benefits, cross training career growth and workplace s

  0%|          | 8/606878 [00:45<1300:08:38,  7.71s/it]

{'id': 'gen-1778000290-z0BnMYOb1F8ov9u7yKtZ', 'object': 'chat.completion', 'created': 1778000290, 'model': 'poolside/laguna-m.1-20260312:free', 'provider': 'Poolside', 'system_fingerprint': None, 'choices': [{'index': 0, 'logprobs': None, 'finish_reason': 'stop', 'native_finish_reason': 'stop', 'message': {'role': 'assistant', 'content': '\n{\n  "benefits": [\n    "Mức lương cạnh tranh & phúc lợi hấp dẫn với đánh giá lương 2 lần/năm (Salary)",\n    "Bảo hiểm xã hội 100% lương (Social Insurance)",\n    "Khám sức khỏe định kỳ hàng năm",\n    "Cấp MacBook hoặc laptop cá nhân để làm việc",\n    "Thưởng cuối năm dựa trên hiệu suất",\n    "Đào tạo ngoại ngữ miễn phí: Tiếng Anh và Tiếng Nhật",\n    "Cơ hội làm việc tại Nhật Bản để phát triển sự nghiệp",\n    "Ẩn ngày sinh nhật hàng tháng cho nhân viên",\n    "Môi trường làm việc chuyên nghiệp, thân thiện, cởi mở",\n    "Hoạt động đội nhóm: Tổng kết tháng, Du lịch công ty, Teambuilding",\n    "Quà/giúp đỡ đặc biệt cho đám cưới, sinh con, 8/3, 

  0%|          | 8/606878 [00:49<1033:29:13,  6.13s/it]


{'id': 'gen-1778000331-cu3jX4CkK6DTZ1HhhZq8', 'object': 'chat.completion', 'created': 1778000331, 'model': 'poolside/laguna-m.1-20260312:free', 'provider': 'Poolside', 'system_fingerprint': None, 'choices': [{'index': 0, 'logprobs': None, 'finish_reason': 'stop', 'native_finish_reason': 'stop', 'message': {'role': 'assistant', 'content': '\n```json\n{\n  "benefits": [],\n  "requirements": []\n}\n```\n', 'refusal': None, 'reasoning': '\nThe user wants me to extract benefits and requirements from provided text, but both the "Benefits Text" and "Requirements Text" fields show "None". \n\nLet me check the rules:\n1. Return JSON with exactly two keys: "benefits" and "requirements"\n2. Benefits should have salary/compensation first, then SHUI/social insurance second if mentioned\n3. Bullet points should be concise and in Vietnamese\n\nSince there\'s no actual content provided (both are "None"), I should return empty lists for both benefits and requirements.\n\nLet me create the JSON response

In [ ]:
import re

def extract_and_normalize_salary(text):
    """
    Extracts salary numbers and currencies from text and normalizes them.
    Handles formats like: '15 - 35 triệu', '4000 usd', '1000 ~ 1400 USD', '26 - 36 triệu'
    """
    if not text or not isinstance(text, str):
        return None

    text = text.lower().replace(',', '')

    # Regex for range format: 'min - max currency' or 'min ~ max currency'
    range_pattern = r'(\d+(?:\.\d+)?)\s*(?:-|~|đến)\s*(\d+(?:\.\d+)?)\s*(triệu|tr|usd|\$|vnd|vnđ)'

    # Regex for single value: 'value currency'
    single_pattern = r'(\d+(?:\.\d+)?)\s*(triệu|tr|usd|\$|vnd|vnđ)'

    range_match = re.search(range_pattern, text)
    if range_match:
        min_val = float(range_match.group(1))
        max_val = float(range_match.group(2))
        currency = range_match.group(3)
        return normalize_value(min_val, currency), normalize_value(max_val, currency), 'vnd'

    single_match = re.search(single_pattern, text)
    if single_match:
        val = float(single_match.group(1))
        currency = single_match.group(2)
        norm_val = normalize_value(val, currency)
        return norm_val, norm_val, 'vnd'

    return None

def normalize_value(value, currency):
    """
    Normalizes the value based on the currency string.
    Assumes base unit is VND. USD conversion is approx (can be adjusted).
    """
    currency = currency.lower()
    if currency in ['triệu', 'tr']:
        return value * 1_000_000
    elif currency in ['usd', '$']:
        return value * 25_000 # Example conversion rate
    else:
        return value

import pandas as pd

# Apply to all rows in the dataframe
def safe_extract_salary(text):
    res = extract_and_normalize_salary(text)
    if res:
        return pd.Series(res)
    return pd.Series([None, None, None])

print("Applying salary extraction to all rows...")
from tqdm import tqdm
tqdm.pandas()
df[['min_salary', 'max_salary', 'currency']] = df['salary'].progress_apply(safe_extract_salary)
print(df[['salary', 'min_salary', 'max_salary', 'currency']].head())


### Step 3: Knowledge Graph
Let's use Neo4j to extract dataset into a knowledge graph since it's structured.

In [ ]:
pip install neo4j


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from neo4j import GraphDatabase

# 1. Connection Details
URI = "neo4j+s://48099f3d.databases.neo4j.io"
AUTH = ("48099f3d", "Vt2o8fQLxuRgsUN0Bf-4WaRv4zvdM7hMEqCsUMBWgUg")

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()

In [ ]:
def inject_vietnamese_jobs(uri, auth, dataframe, rows_limit=1000):
    # 1. Limit the data immediately
    # We slice the dataframe (or records) to the first 'limit' rows
    records = dataframe.head(rows_limit).to_dict(orient='records')

    print(f"Starting injection for {len(records)} records...")

    with GraphDatabase.driver(uri, auth=auth) as driver:
        # Create Constraints
        driver.execute_query("CREATE CONSTRAINT IF NOT EXISTS FOR (j:Job) REQUIRE j.id IS UNIQUE")
        driver.execute_query("CREATE CONSTRAINT IF NOT EXISTS FOR (c:Company) REQUIRE c.name IS UNIQUE")
        driver.execute_query("CREATE CONSTRAINT IF NOT EXISTS FOR (l:Location) REQUIRE l.name IS UNIQUE")

        query = """
        UNWIND $batch AS row
        MERGE (j:Job {id: row.id})
        SET j.title = row.job_title,
            j.salary = row.salary,
            j.experience = row.experience_level,
            j.description = row.job_description

        WITH j, row WHERE row.company_name IS NOT NULL
        MERGE (c:Company {name: row.company_name})
        MERGE (j)-[:POSTED_BY]->(c)

        WITH j, row WHERE row.location IS NOT NULL
        MERGE (l:Location {name: row.location})
        MERGE (j)-[:LOCATED_IN]->(l)
        """

        # 2. Process the limited set in batches
        batch_size = 500
        for i in range(0, len(records), batch_size):
            batch = records[i:i + batch_size]
            driver.execute_query(query, batch=batch)
            print(f"Injected {i + len(batch)} / {len(records)} records...", end='\r')

    print("\nInjection complete.")



In [ ]:
# To run it for just the first 100 rows:
inject_vietnamese_jobs(URI, AUTH, df, rows_limit=100)

Starting injection for 100 records...
Injected 100 / 100 records...
Injection complete.
